# Retail Data Analytics with PySpark
## Setup and Data Ingestion

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *

file_path = "file:/Workspace/Users/enascimento@myseneca.ca/online_retail_II.csv"


df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(file_path)
)

display(df.limit(10))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom


## Data Cleaning and Standardization
Standardizing data types and mapping original CSV column names to project-standard names.

In [0]:


df = (
    df.withColumn("invoice_no", F.col("Invoice").cast("string"))
      .withColumn("stock_code", F.col("StockCode").cast("string"))
      .withColumn("description", F.col("Description").cast("string"))
      .withColumn("quantity", F.col("Quantity").cast("int"))
      .withColumn("invoice_date", F.col("InvoiceDate").cast("timestamp"))
      .withColumn("unit_price", F.col("Price").cast("double"))
      .withColumn("customer_id", F.col("Customer ID").cast("int"))
      .withColumn("country", F.col("Country").cast("string"))
      .dropna(subset=["Description"]) # Drop rows where the original Description is missing
)

df = (
    df.withColumn("sales_amount", F.col("quantity") * F.col("unit_price"))
      .withColumn("year_month", F.date_format("invoice_date", "yyyyMM").cast("int"))
      .withColumn("month_year", F.date_format("invoice_date", "MMM yyyy"))
)

display(df.limit(5))

Invoice,StockCode,description,quantity,InvoiceDate,Price,Customer ID,country,invoice_no,stock_code,invoice_date,unit_price,customer_id,sales_amount,year_month,month_year
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom,489434,85048,2009-12-01T07:45:00.000Z,6.95,13085,83.4,200912,Dec 2009
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,489434,79323P,2009-12-01T07:45:00.000Z,6.75,13085,81.0,200912,Dec 2009
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,489434,79323W,2009-12-01T07:45:00.000Z,6.75,13085,81.0,200912,Dec 2009
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom,489434,22041,2009-12-01T07:45:00.000Z,2.1,13085,100.80000000000001,200912,Dec 2009
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,489434,21232,2009-12-01T07:45:00.000Z,1.25,13085,30.0,200912,Dec 2009


## Null Check
Checking for missing values across all columns.

In [0]:
null_check_df = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_check_df)

Invoice,StockCode,description,quantity,InvoiceDate,Price,Customer ID,country,invoice_no,stock_code,invoice_date,unit_price,customer_id,sales_amount,year_month,month_year
0,0,0,0,0,0,238625,0,0,0,0,0,238625,0,0,0


## Total Invoice Amount Distribution
Calculating descriptive statistics for positive transactions.

In [0]:
df_positive = df.filter((F.col("quantity") > 0) & (F.col("unit_price") > 0))

invoice_amount_df = (
    df_positive
    .groupBy("invoice_no")
    .agg(F.round(F.sum("sales_amount"), 2).alias("invoice_amount"))
    .orderBy("invoice_amount")
)

invoice_amount_df.select(
    F.min("invoice_amount").alias("min_invoice_amount"),
    F.max("invoice_amount").alias("max_invoice_amount"),
    F.mean("invoice_amount").alias("mean_invoice_amount"),
    F.expr("percentile(invoice_amount, 0.5)").alias("median_invoice_amount")
).show()

+------------------+------------------+-------------------+---------------------+
|min_invoice_amount|max_invoice_amount|mean_invoice_amount|median_invoice_amount|
+------------------+------------------+-------------------+---------------------+
|              0.19|          168469.6|  523.3037606666999|              304.315|
+------------------+------------------+-------------------+---------------------+



## Monthly Placed and Canceled Orders

In [0]:
# Monthly canceled orders (starting with 'C')
monthly_canceled_orders_df = (
    df.filter(F.col("invoice_no").startswith("C"))
      .groupBy("year_month", "month_year")
      .agg(F.countDistinct("invoice_no").alias("canceled_orders"))
)


monthly_total_orders_df = (
    df.groupBy("year_month", "month_year")
      .agg(F.countDistinct("invoice_no").alias("total_orders"))
)


monthly_orders_df = (
    monthly_total_orders_df.join(
        monthly_canceled_orders_df,
        on=["year_month", "month_year"],
        how="left"
    )
    .fillna(0, subset=["canceled_orders"])
    .withColumn("canceled_orders", F.col("canceled_orders").cast("int"))
    .withColumn("placed_orders", F.col("total_orders") - 2 * F.col("canceled_orders"))
    .select("year_month", "month_year", "placed_orders", "canceled_orders")
    .orderBy("year_month")
)

display(monthly_orders_df)

year_month,month_year,placed_orders,canceled_orders
200912,Dec 2009,1300,401
201001,Jan 2010,811,300
201002,Feb 2010,979,240
201003,Mar 2010,1301,407
201004,Apr 2010,1174,304
201005,May 2010,1128,407
201006,Jun 2010,1327,357
201007,Jul 2010,1244,344
201008,Aug 2010,1172,273
201009,Sep 2010,1498,371


In [0]:
# Filter for actual sales
monthly_sales_df = (
    df.filter((F.col("quantity") > 0) & (F.col("unit_price") > 0))
      .groupBy("year_month", "month_year")
      .agg(F.round(F.sum("sales_amount"), 2).alias("monthly_sales"))
      .orderBy("year_month")
)

sales_window = Window.orderBy("year_month")

monthly_sales_growth_df = (
    monthly_sales_df
      .withColumn("previous_month_sales", F.lag("monthly_sales").over(sales_window))
      .withColumn(
          "growth_rate",
          F.round(((F.col("monthly_sales") - F.col("previous_month_sales")) / F.col("previous_month_sales")) * 100, 2)
      )
      .orderBy("year_month")
)

display(monthly_sales_growth_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


year_month,month_year,monthly_sales,previous_month_sales,growth_rate
200912,Dec 2009,825685.76,null,null
201001,Jan 2010,652708.5,825685.76,-20.95
201002,Feb 2010,553713.31,652708.5,-15.17
201003,Mar 2010,833570.13,553713.31,50.54
201004,Apr 2010,681528.99,833570.13,-18.24
201005,May 2010,659858.86,681528.99,-3.18
201006,Jun 2010,752270.14,659858.86,14.0
201007,Jul 2010,650712.94,752270.14,-13.5
201008,Aug 2010,697274.91,650712.94,7.16
201009,Sep 2010,924333.01,697274.91,32.56


## RFM Analysis
Calculating Recency, Frequency, and Monetary scores.

In [0]:
# RFM Base
rfm_base_df = df.filter(F.col("customer_id").isNotNull())
max_date = rfm_base_df.select(F.max("invoice_date")).collect()[0][0]
snapshot_date = F.date_add(F.lit(max_date), 1)

rfm_df = (
    rfm_base_df.groupBy("customer_id")
      .agg(
          F.max("invoice_date").alias("last_purchase_date"),
          F.countDistinct("invoice_no").alias("frequency"),
          F.round(F.sum("sales_amount"), 2).alias("monetary")
      )
      .withColumn("recency", F.datediff(F.lit(snapshot_date), F.col("last_purchase_date")))
)

# Quintiles for scoring
r_q = rfm_df.approxQuantile("recency", [0.2, 0.4, 0.6, 0.8], 0.01)
f_q = rfm_df.approxQuantile("frequency", [0.2, 0.4, 0.6, 0.8], 0.01)
m_q = rfm_df.approxQuantile("monetary", [0.2, 0.4, 0.6, 0.8], 0.01)

# Scoring and Segmentation
rfm_segmented_df = (
    rfm_df.withColumn("r_score", F.when(F.col("recency") <= r_q[0], 5).when(F.col("recency") <= r_q[1], 4).when(F.col("recency") <= r_q[2], 3).when(F.col("recency") <= r_q[3], 2).otherwise(1))
    .withColumn("f_score", F.when(F.col("frequency") <= f_q[0], 1).when(F.col("frequency") <= f_q[1], 2).when(F.col("frequency") <= f_q[2], 3).when(F.col("frequency") <= f_q[3], 4).otherwise(5))
    .withColumn("m_score", F.when(F.col("monetary") <= m_q[0], 1).when(F.col("monetary") <= m_q[1], 2).when(F.col("monetary") <= m_q[2], 3).when(F.col("monetary") <= m_q[3], 4).otherwise(5))
    .withColumn("segment", 
        F.when((F.col("r_score") >= 4) & (F.col("f_score") >= 4) & (F.col("m_score") >= 4), "Champions")
         .when((F.col("r_score") >= 3) & (F.col("f_score") >= 3) & (F.col("m_score") >= 3), "Loyal Customers")
         .when((F.col("r_score") <= 2) & (F.col("f_score") >= 3), "At Risk")
         .when((F.col("r_score") <= 2) & (F.col("f_score") <= 2), "Lost")
         .otherwise("Others"))
)

display(rfm_segmented_df.groupBy("segment").count().orderBy(F.desc("count")))

segment,count
Lost,1800
Others,1275
Champions,1261
Loyal Customers,1006
At Risk,600
